IMPORTANT!

Change the filename write away with your first and last name!


Also, just in case, put your name here:

- First name:
- Last name:

# Instruction

Some evolutionary scientists argue that costly punishment is unnecessary to stabilize cooperation, and that cheaper sanctions of defection suffice (e.g., reputational sanctions).


Is inter-group competition necessary to stabilize cooperation in this case?


Think of an answer and test it.

**Tips:** group selection refers to the *epsilon* variable; "cheap sanction" refers to low cost of punishment, related to the *k* variable (see table).

| Parameter | Description                        | Default|
|-----------|------------------------------------|--------|
| N         | Number of groups                   | 128    |
| n         | Agents per group                   | 4     |
| t_max     | Number of time steps               | 1000   |
| c         | Cost of cooperation                | 0.2    |
| m         | Rate of between-group mixing       | 0.01   |
| e         | Cooperation error rate             | 0.02   |
| p         | Punishment cost to defector        | 0.8    |
| k         | Punishment cost to punisher        | 0.2    |
| mu        | Mutation rate                      | 0.01   |
| epsilon   | Frequency of inter-group conflict  | 0.015  |

# Code

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def initialise_agents(n, N, random=False):
    """A0. Create the agent matrix with the standard initial conditions.
    
    - If random=False (default): group 0 = all P, group N-1 = all C, others = all D
    - If random=True: each agent is assigned C, D, or P with equal probability (1/3 each)

    Parameters
    ----------
    n      (int) : number of agents per group
    N      (int) : number of groups
    random (bool): if True, assign strategies randomly; default is False

    Returns
    -------
    agent: np.ndarray of shape (n, N)
    """
    agent = np.empty((n, N), dtype=str)

    if random:
        # Each agent independently draws from {C, D, P} with equal probability
        agent = np.random.choice(["C", "D", "P"], size=(n, N))
    else:
        # Standard initial conditions from Boyd et al. 2003
        agent[:, 0] = "P"
        agent[:, 1:] = "D"
    return agent


def initialise_payoffs(n, N):
    """A1a. Create a payoff matrix filled with the baseline payoff of 1.

    Parameters
    ----------
    n (int): number of agents per group
    N (int): number of groups

    Returns
    -------
    payoff: np.ndarray of shape (n, N)
    """
    return np.ones((n, N), dtype=float)


def apply_cooperation_costs(agent, payoff, c, e=0):
    """A1b. Reduce the payoff of every contributor (C or P) by c.

    Contributors are C or P agents who cooperate with probability 1-e.
    The parameter e represents cooperation errors: a C or P agent 
    accidentally defects with probability e.

    Parameters
    ----------
    agent (np.ndarray of shape (n, N)) : strategy matrix
    payoff (np.ndarray of shape (n, N)): payoff matrix
    c (float)                          : cost of cooperation
    e (float)                          : cooperation error rate

    Returns
    -------
    payoff (np.ndarray)      : updated payoff matrix
    contributors (np.ndarray): boolean mask of contributing agents
    """
    # Random draw to determine whether each C/P agent cooperates this round
    contribute = np.random.uniform(size=agent.shape)

    # Contributors are C or P agents who clear the 1-e threshold
    contributors = ((agent == "C") | (agent == "P")) & (contribute > e)

    payoff[contributors] -= c
    return payoff, contribute


def apply_punishment(agent, payoff, contribute, p, k, e = 0):
    """A2. Apply punishment costs within groups that contain at least one Punisher.

    For each Punisher i in group j:
      - Punisher's payoff is reduced by (k/n) * number_of_defectors_in_group
      - Each defector's payoff in the group is reduced by p/n

    Parameters
    ----------
    agent  (np.ndarray of shape (n, N)): strategy matrix
    payoff (np.ndarray of shape (n, N)): payoff matrix (modified in-place)
    p (float)                          : cost inflicted on each defector
    k (float)                          : cost paid by each punisher per defector
    e (float)                          : cooperation error rate

    Returns
    -------
    payoff (np.ndarray): updated payoff matrix
    """
    n = agent.shape[0]

    # Identify groups that contain at least one punisher
    groups_with_punishers = np.unique(np.where(agent == "P")[1])

    # Boolean mask: True where agent is a Defector
    defections = (agent == "D") | (((agent == "C") | (agent == "P")) & (contribute <= e))

    for j in groups_with_punishers:
        for i in range(n):
            if agent[i, j] == "P":

                # Boolean mask: all agents in this group except agent i
                other_rows = np.arange(n) != i

                # Number of defectors in the group, excluding the punisher itself
                n_defectors = np.sum(defections[other_rows, j])

                # Punisher pays k/n per defector
                payoff[i, j] -= n_defectors * k / n

                # Indices of those defectors
                defector_indices = np.where(other_rows & defections[:, j])[0]

                # Each defector receives a penalty of p/n
                payoff[defector_indices, j] -= p / n

    return payoff


def social_learning(agent, payoff, m):
    """A3. Payoff-biased social learning across and within groups.

    Each agent selects a demonstrator (within-group with prob 1-m,
    between-group with prob m) and copies their strategy with
    probability W = payoff_dem / (payoff_dem + payoff_agent).

    Parameters
    ----------
    agent (np.ndarray of shape (n, N)) : strategy matrix
    payoff (np.ndarray of shape (n, N)): payoff matrix
    m (float): probability of choosing a between-group demonstrator

    Returns
    -------
    agent (np.ndarray): updated strategy matrix
    """
    n, N = agent.shape

    # Snapshot of strategies before learning begins, to avoid overlap:
    # an agent updated early in the loop must not influence agents updated later
    previous_agent = agent.copy()

    for j in range(N):
        for i in range(n):

            # --- Select demonstrator ---
            if np.random.uniform() > m:
                # Within-group: pick any agent in the same group, excluding self
                same_group_others = np.arange(n)[np.arange(n) != i]
                dem_row = np.random.choice(same_group_others)
                dem_col = j
            else:
                # Between-group: pick any agent from a different group
                other_groups = np.arange(N)[np.arange(N) != j]
                dem_row = np.random.choice(np.arange(n))
                dem_col = np.random.choice(other_groups)

            # --- Compute copying probability W ---
            dem_payoff   = payoff[dem_row, dem_col]
            agent_payoff = payoff[i, j]
            W = dem_payoff / (dem_payoff + agent_payoff)

            # --- Copy demonstrator's strategy with probability W ---
            if np.random.uniform() < W:
                agent[i, j] = previous_agent[dem_row, dem_col]

    return agent


def group_selection(agent, contribute, epsilon, e=0):
    """A4. Inter-group conflict and group selection.

    Groups are randomly paired. Each contest is kept with probability epsilon.
    The group with fewer defectors is more likely to win and replace the loser.

    Parameters
    ----------
    agent (np.ndarray of shape (n, N)): strategy matrix
    epsilon (float): probability that each potential contest actually takes place

    Returns
    -------
    agent (np.ndarray): updated strategy matrix
    """
    n, N = agent.shape

    # --- Create random pairings of all N groups ---
    shuffled = np.random.permutation(N)
    contests = pd.DataFrame(shuffled.reshape(N // 2, 2), columns=["group1", "group2"])

    # --- Keep only a fraction of contests, determined by epsilon ---
    contests = contests[np.random.uniform(size=N // 2) < epsilon].reset_index(drop=True)

    # --- Recompute defectors at this point in time ---
    defections = (agent == "D") | (((agent == "C") | (agent == "P")) & (contribute <= e))

    # --- Only run if at least one contest survived the epsilon filter ---
    if len(contests) > 0:

        for i in range(len(contests)):

            group1 = int(contests.loc[i, "group1"])
            group2 = int(contests.loc[i, "group2"])

            # Proportion of defectors in each competing group
            d1 = np.sum(defections[:, group1]) / n
            d2 = np.sum(defections[:, group2]) / n

            # Probability that group1 wins (Boyd et al. 2003)
            # More defectors in group2 → higher chance group1 wins
            d = 0.5 + (d2 - d1) / 2

            if np.random.uniform() < d:
                # Group 1 wins: group 2 is fully replaced by group 1
                agent[:, group2] = agent[:, group1]
            else:
                # Group 2 wins: group 1 is fully replaced by group 2
                agent[:, group1] = agent[:, group2]

    return agent


def mutation(agent, mu=0.5):
    """A.5 Mutation
    Mutate agent strategies with probability mu.

    Parameters
    ----------
    agent (np.ndarray of shape (n, N)): strategy matrix
    mu (float): mutation probability

    Returns
    -------
    agent (np.ndarray): updated strategy matrix
    """
    n, N = agent.shape

    # Flattened mutation draws
    mutate = np.random.uniform(size=n * N).reshape(n, N)

    # Snapshot to avoid overlap
    previous_agent = agent.copy()

    # --- Mutate D agents → {P, C} ---
    mask_D = (mutate < mu) & (previous_agent == "D")
    agent[mask_D] = np.random.choice(["P", "C"], size=mask_D.sum(), replace=True)

    # --- Mutate C agents → {P, D} ---
    mask_C = (mutate < mu) & (previous_agent == "C")
    agent[mask_C] = np.random.choice(["P", "D"], size=mask_C.sum(), replace=True)

    # --- Mutate P agents → {D, C} ---
    mask_P = (mutate < mu) & (previous_agent == "P")
    agent[mask_P] = np.random.choice(["D", "C"], size=mask_P.sum(), replace=True)

    return agent


def cooperation(N=128,
                n=4,
                t_max=1000,
                c=0.2,
                m=0.01,
                e=0.02,
                p=0.8,
                k=0.2,
                epsilon=0.015,
                mu=0.01,
                show_plot=True):

    """Run the cultural group selection simulation (Boyd et al. 2003).

    At each time step:
      1. Contributors (C, P) pay cooperation cost c
      2. Punishers pay k per defector and inflict cost p on each defector
      3. Agents copy higher-payoff demonstrators (within- or between-group)
      4. Groups compete; the losing group is fully replaced by the winner

    Parameters
    ----------
    N (int)         : number of groups
    n (int)         : agents per group
    t_max (int)     : number of time steps
    c (float)       : cost of cooperation
    m (float)       : between-group mixing rate
    e (float)       : cooperation error rate
    p (float)       : punishment cost inflicted on each defector
    k (float)       : punishment cost paid by each punisher per defector
    mu (float)      : mutation rate
    epsilon (float) : frequency of inter-group conflict
    show_plot (bool): if True, plot cooperation and punishment frequencies

    Returns
    -------
    dict with keys:
      'agent'     : final agent matrix (np.ndarray)
      'output'    : pd.DataFrame with columns PandC and P over time
      'mean_coop' : mean cooperation frequency over the last 50% of time steps
    """

    # --- Initialise agent matrix and output tracker ---
    agent = initialise_agents(n, N)
    output = pd.DataFrame({"PandC": np.full(t_max, np.nan),
                           "P":     np.full(t_max, np.nan)})

    # Record frequencies at t = 0 (initial state)
    output.loc[0, "PandC"] = np.sum((agent == "C") | (agent == "P")) / (N * n)
    output.loc[0, "P"]     = np.sum(agent == "P") / (N * n)

    for t in range(1, t_max):

        # 1. Cooperation
        payoff = initialise_payoffs(n, N)
        payoff, contribute = apply_cooperation_costs(agent, payoff, c, e)

        # 2. Punishment
        payoff = apply_punishment(agent, payoff, contribute, p, k, e)

        # 3. Social learning
        agent = social_learning(agent, payoff, m)

        # 4. Group selection
        agent = group_selection(agent, contribute, epsilon, e)

        # 5. Mutation
        agent = mutation(agent, mu)

        # Record cooperation and punishment frequencies for this time step
        output.loc[t, "PandC"] = np.sum((agent == "C") | (agent == "P")) / (N * n)
        output.loc[t, "P"]     = np.sum(agent == "P") / (N * n)

    # --- Optional plot ---
    if show_plot:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(range(t_max), output["PandC"], linestyle="-",      label="C and P (all contributors)")
        ax.plot(range(t_max), output["P"],     linestyle="dotted", label="P only (punishers)")
        ax.set_ylim(0, 1)
        ax.set_xlabel("Generation")
        ax.set_ylabel("Frequency")
        ax.set_title("Cultural group selection - Boyd et al. 2003")
        ax.legend()
        plt.tight_layout()
        plt.show()

    # Mean cooperation over the second half of the simulation
    mean_coop = output["PandC"].iloc[t_max // 2:].mean()

    return {"agent": agent, "output": output, "mean_coop": mean_coop}

# Answer

Keep n at 32

In [ ]:
simulations = cooperation(n = 32)

print(f"Mean cooperation (last 50% of time steps): {simulations['mean_coop']:.3f}")

You can write the answer to the questions here